# Supply Chain Control Tower — cleaning & EDA

Sanity checks on the star-schema parquet before the Power BI control tower.

Focus: revenue flags, OTIF / perfect-order rates, freight & CO2, inventory on-hand.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data')
OUT = Path('outputs')
OUT.mkdir(parents=True, exist_ok=True)

orders = pd.read_parquet(DATA / 'fact_orders.parquet')
ship = pd.read_parquet(DATA / 'fact_shipments.parquet')
print('orders', len(orders), 'shipments', len(ship))
orders.head()


In [ ]:
rev = orders[orders['is_revenue'] == True]
if rev.empty:
    rev = orders[orders['is_revenue'] == 1]
o = orders.groupby('order_id').agg(otif=('is_otif', 'min'), perfect=('is_perfect_order', 'min'))
print('revenue_m', round(rev['net_sales'].sum() / 1e6, 1))
print('orders', orders['order_id'].nunique())
print('otif_pct', round(100 * o['otif'].mean(), 1))
print('perfect_pct', round(100 * o['perfect'].mean(), 1))
print('freight_m', round(ship['freight_cost'].sum() / 1e6, 1))
print('co2_t', round(ship['co2_kg'].sum() / 1000, 1))


In [ ]:
if 'ship_mode' in ship.columns:
    mix = ship.groupby('ship_mode').size().sort_values()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(mix.index.astype(str), mix.values)
    ax.set_title('Shipments by mode')
    fig.tight_layout()
    fig.savefig(OUT / 'shipments_by_mode.png', dpi=120)
    plt.show()

inv_path = DATA / 'fact_inventory.parquet'
if inv_path.exists():
    inv = pd.read_parquet(inv_path)
    col = 'on_hand_qty' if 'on_hand_qty' in inv.columns else [c for c in inv.columns if 'on_hand' in c.lower() or 'qty' in c.lower()]
    if isinstance(col, list):
        col = col[0] if col else None
    if col:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.hist(inv[col].dropna(), bins=40)
        ax.set_title('Inventory on-hand distribution')
        fig.tight_layout()
        fig.savefig(OUT / 'inventory_onhand.png', dpi=120)
        plt.show()
